# Azose & Raftery (2019) — Final Model Fit

**All countries · All periods (1990–2010) · No hold-out**

This notebook fits the complete hierarchical Bayesian model for migration flows:
- **Outflow model**: Emigration rates (log δ_it) with AR(1) dynamics
- **Inflow model**: Destination shares via multinomial-softmax

Posteriors are saved to `posteriors_final/`

In [1]:
import os
import time
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

print("Dependencies loaded successfully.")

Dependencies loaded successfully.


In [2]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Sampling parameters
N_CHAINS = 4
N_WARMUP = 500
N_SAMPLES = 500
SEED = 42

# Paths
PROJ = Path.cwd()
DATA_FLOWS = PROJ / "data" / "azoseRaftery2019flows.csv"
DATA_GRAVITY = PROJ / "data_final" / "FINAL_GRAVITY_TRAINING_MATRIX.csv"
OUTFLOW_STAN = PROJ / "paper_outflow.stan"
INFLOW_STAN = PROJ / "paper_inflow.stan"
SAVE_DIR = PROJ / "posteriors_final"

# Hyperparameters (paper Table 3)
MU_0 = -5.0
A_0 = 2.0
B_0 = 5.0
P_0 = 2.0
Q_0 = 5.0

print(f"Configuration:")
print(f"  Chains: {N_CHAINS}")
print(f"  Warmup: {N_WARMUP}")
print(f"  Samples: {N_SAMPLES}")
print(f"  Project: {PROJ}")
print(f"  Save dir: {SAVE_DIR}")

Configuration:
  Chains: 4
  Warmup: 500
  Samples: 500
  Project: /home/onyxia/work/ProjetStat
  Save dir: /home/onyxia/work/ProjetStat/posteriors_final


## 1. Data Preparation

Load migration flows, join with population data, compute emigration rates and rates, build country and corridor indexing for Stan.

In [3]:
def prepare_data():
    print("=" * 70)
    print("DATA PREPARATION — FINAL (all countries, 1990–2010)")
    print("=" * 70)

    df = pd.read_csv(DATA_FLOWS)
    df = df[df['origIso'] != df['destIso']].copy()

    # --- Population ---
    grav = pd.read_csv(DATA_GRAVITY)
    pop = grav[['iso3_o', 'year', 'pop_o']].drop_duplicates(['iso3_o', 'year'])
    pop.columns = ['origIso', 'year', 'pop']

    df = df.merge(pop, on=['origIso', 'year'], how='left')
    for c in df['origIso'].unique():
        mask = (df['origIso'] == c) & df['pop'].isna()
        if mask.any():
            med = df.loc[df['origIso'] == c, 'pop'].median()
            df.loc[mask, 'pop'] = med if not pd.isna(med) else 1e6

    # --- Total emigration & rate ---
    E = df.groupby(['origIso', 'year'])['migrantCount'].sum().reset_index()
    E.columns = ['origIso', 'year', 'E_it']
    E = E.merge(pop, on=['origIso', 'year'], how='left')
    E['pop'] = E['pop'].fillna(E.groupby('origIso')['pop'].transform('median'))
    E['pop'] = E['pop'].fillna(1e6)
    E['delta'] = E['E_it'] / E['pop']
    E.loc[E['delta'] <= 0, 'delta'] = 1e-10
    E['log_delta'] = np.log(E['delta'])

    # --- Country & corridor indexing ---
    countries = sorted(set(df['origIso'].unique()) | set(df['destIso'].unique()))
    c2i = {c: i + 1 for i, c in enumerate(countries)}  # 1-indexed for Stan

    pos = df[df['migrantCount'] > 0].copy()
    pos['pair'] = pos['origIso'] + '_' + pos['destIso']
    corridors = sorted(pos['pair'].unique())
    corr2i = {c: i + 1 for i, c in enumerate(corridors)}
    corr_origin = [c2i[c.split('_')[0]] for c in corridors]

    print(f"Countries: {len(countries)}")
    print(f"Corridors (>0 flow): {len(corridors):,}")
    print(f"Total obs: {len(df):,}")
    print(f"Positive obs: {len(pos):,}")
    print(f"Periods: {sorted(df['year'].unique())}")
    print(f"Emigration rate — mean: {E['delta'].mean():.4f}, "
          f"mean log δ: {E['log_delta'].mean():.2f}")

    return {
        'df': df, 'E': E,
        'countries': countries, 'c2i': c2i,
        'corridors': corridors, 'corr2i': corr2i, 'corr_origin': corr_origin,
        'pop': pop,
    }

## 2. Build Outflow Stan Data

Structure emigration rates for the outflow model: initial observations (first period per country) and AR(1) observations (5-year lags).

In [4]:
def build_outflow_data(data):
    print("\n--- Building OUTFLOW model data ---")

    E = data['E']
    c2i = data['c2i']

    E = E.sort_values(['origIso', 'year']).reset_index(drop=True)

    init_origin, init_ld = [], []
    ar_origin, ar_ld, ar_lag = [], [], []

    for orig, grp in E.groupby('origIso'):
        oidx = c2i[orig]
        years = grp['year'].values
        lds = grp['log_delta'].values

        init_origin.append(oidx)
        init_ld.append(lds[0])

        for i in range(1, len(grp)):
            if years[i] - years[i - 1] == 5:
                ar_origin.append(oidx)
                ar_ld.append(lds[i])
                ar_lag.append(lds[i - 1])
            else:
                init_origin.append(oidx)
                init_ld.append(lds[i])

    print(f"  Init: {len(init_origin)} | AR: {len(ar_origin)}")

    stan_data = {
        'N_orig': len(data['countries']),
        'N_init': len(init_origin),
        'init_origin': np.array(init_origin, dtype=int),
        'log_delta_init': np.array(init_ld),
        'N_ar': len(ar_origin),
        'ar_origin': np.array(ar_origin, dtype=int),
        'log_delta_ar': np.array(ar_ld),
        'lag_log_delta': np.array(ar_lag),
        # No prediction — set to empty
        'N_pred': 0,
        'pred_origin': np.array([], dtype=int),
        'pred_lag_log_delta': np.array([]),
        'mu_0': MU_0,
        'a_0': A_0,
        'b_0': B_0,
    }
    return stan_data

## 3. Build Inflow Stan Data

Structure flows into grouped and flattened arrays for the multinomial-softmax inflow model. Each (origin, time) group contains corridor flows that sum to total emigration.

In [5]:
def build_inflow_data(data):
    print("\n--- Building INFLOW model data ---")

    df = data['df']
    corr2i = data['corr2i']
    corr_origin = data['corr_origin']

    pos = df[df['migrantCount'] > 0].copy()
    pos['pair'] = pos['origIso'] + '_' + pos['destIso']
    pos = pos[pos['pair'].isin(corr2i)].copy()

    groups = []
    flat_corridor = []
    flat_count = []

    for (orig, year), grp in pos.groupby(['origIso', 'year']):
        corridors_in_grp = []
        counts_in_grp = []
        for _, row in grp.iterrows():
            cidx = corr2i[row['pair']]
            corridors_in_grp.append(cidx)
            counts_in_grp.append(int(row['migrantCount']))

        start = len(flat_corridor) + 1
        groups.append({
            'size': len(corridors_in_grp),
            'start': start,
            'N_it': grp['migrantCount'].sum(),
        })
        flat_corridor.extend(corridors_in_grp)
        flat_count.extend(counts_in_grp)

    N_flat = len(flat_corridor)
    print(f"  Groups: {len(groups):,} | Flat obs: {N_flat:,}")

    stan_data = {
        'N_orig': len(data['countries']),
        'N_corridors': len(data['corridors']),
        'N_obs_groups': len(groups),
        'corridor_origin': np.array(corr_origin, dtype=int),
        'group_size': np.array([g['size'] for g in groups], dtype=int),
        'group_start': np.array([g['start'] for g in groups], dtype=int),
        'group_N': np.array([g['N_it'] for g in groups], dtype=int),
        'N_flat': N_flat,
        'flat_corridor': np.array(flat_corridor, dtype=int),
        'flat_count': np.array(flat_count, dtype=int),
        # No prediction — set to empty
        'N_pred_groups': 0,
        'pred_group_size': np.array([], dtype=int),
        'pred_group_start': np.array([], dtype=int),
        'N_pred_flat': 0,
        'pred_flat_corridor': np.array([], dtype=int),
        'p_0': P_0,
        'q_0': Q_0,
    }
    return stan_data

## 4. Compile and Fit Stan Models

Fit both outflow (emigration rates) and inflow (destination shares) models using HMC sampling.

In [6]:
def fit_stan(stan_file, stan_data, label):
    from cmdstanpy import CmdStanModel

    print(f"\n{'=' * 70}")
    print(f"FITTING: {label}")
    print(f"{'=' * 70}")

    model = CmdStanModel(stan_file=str(stan_file))
    print("Compiled.")

    t0 = time.time()
    fit = model.sample(
        data=stan_data,
        chains=N_CHAINS,
        iter_warmup=N_WARMUP,
        iter_sampling=N_SAMPLES,
        seed=SEED,
        show_console=False,
        adapt_delta=0.95,
        max_treedepth=12,
        threads_per_chain=4,
        inits=0,
    )
    elapsed = time.time() - t0
    print(f"Done in {elapsed:.0f}s ({elapsed / 60:.1f} min)")
    print(fit.diagnose())

    return fit

## 5. Save Posteriors and Report Results

Extract posterior samples from fits, compute diagnostics, and save to output directory.

In [7]:
def save_and_report(fit_out, fit_in, data):
    print(f"\n{'=' * 70}")
    print("POSTERIORS & DIAGNOSTICS")
    print(f"{'=' * 70}")

    # --- Outflow diagnostics ---
    phi = fit_out.stan_variable('phi')
    mu_draws = fit_out.stan_variable('mu')
    sigma_draws = fit_out.stan_variable('sigma')
    nu = fit_out.stan_variable('nu')
    tau_0 = fit_out.stan_variable('tau_0')

    print("\n  OUTFLOW MODEL:")
    print(f"    φ  = {phi.mean():.3f} [{np.percentile(phi, 2.5):.3f}, "
          f"{np.percentile(phi, 97.5):.3f}]")
    print(f"    ν  = {nu.mean():.3f} ± {nu.std():.3f}")
    print(f"    τ₀ = {tau_0.mean():.3f} ± {tau_0.std():.3f}")
    print(f"    σ_i mean = {sigma_draws.mean():.3f}, range "
          f"[{sigma_draws.mean(axis=0).min():.3f}, {sigma_draws.mean(axis=0).max():.3f}]")
    print(f"    μ_i mean = {mu_draws.mean():.3f}, SD across countries = "
          f"{mu_draws.mean(axis=0).std():.3f}")

    # --- Inflow diagnostics ---
    kappa = fit_in.stan_variable('kappa')
    psi = fit_in.stan_variable('psi')

    print("\n  INFLOW MODEL:")
    print(f"    κ_ij mean = {kappa.mean():.3f}, SD across corridors = "
          f"{kappa.mean(axis=0).std():.3f}")
    print(f"    ψ_ij mean = {psi.mean():.3f}, range "
          f"[{psi.mean(axis=0).min():.3f}, {psi.mean(axis=0).max():.3f}]")

    # --- Save posteriors ---
    os.makedirs(SAVE_DIR, exist_ok=True)

    np.savez(SAVE_DIR / 'outflow_posteriors.npz',
             phi=phi, mu=mu_draws, sigma=sigma_draws,
             nu=nu, tau_0=tau_0)
    np.savez(SAVE_DIR / 'inflow_posteriors.npz',
             kappa=kappa, psi=psi)

    meta = {'countries': data['countries'], 'corridors': data['corridors']}
    with open(SAVE_DIR / 'metadata.json', 'w') as f:
        json.dump(meta, f)

    print(f"\n  Posteriors saved to {SAVE_DIR}/")
    print(f"    outflow_posteriors.npz  — phi{phi.shape}, mu{mu_draws.shape}, "
          f"sigma{sigma_draws.shape}")
    print(f"    inflow_posteriors.npz   — kappa{kappa.shape}, psi{psi.shape}")
    print(f"    metadata.json           — {len(data['countries'])} countries, "
          f"{len(data['corridors']):,} corridors")

## 6. Execute Full Pipeline

Run all steps in sequence: prepare data, build Stan inputs, fit both models, save posteriors.

In [8]:
print("\n" + "=" * 70)
print("  AZOSE & RAFTERY (2019) — FINAL FIT")
print("  All countries · All periods (1990–2010)")
print("=" * 70)

# 1. Prepare data
data = prepare_data()

# 2. Build Stan inputs
outflow_data = build_outflow_data(data)
inflow_data = build_inflow_data(data)

# 3. Fit models
fit_out = fit_stan(OUTFLOW_STAN, outflow_data, "OUTFLOW — all countries")
fit_in = fit_stan(INFLOW_STAN, inflow_data, "INFLOW — all corridors")

# 4. Save and report
save_and_report(fit_out, fit_in, data)

print(f"\n{'=' * 70}")
print("  DONE — Final model fitted on all data.")
print(f"{'=' * 70}")


  AZOSE & RAFTERY (2019) — FINAL FIT
  All countries · All periods (1990–2010)
DATA PREPARATION — FINAL (all countries, 1990–2010)
Countries: 200
Corridors (>0 flow): 23,607
Total obs: 199,000
Positive obs: 94,027
Periods: [np.int64(1990), np.int64(1995), np.int64(2000), np.int64(2005), np.int64(2010)]
Emigration rate — mean: 0.0466, mean log δ: -3.81

--- Building OUTFLOW model data ---
  Init: 200 | AR: 800

--- Building INFLOW model data ---


05:54:13 - cmdstanpy - INFO - CmdStan start processing


  Groups: 1,000 | Flat obs: 94,027

FITTING: OUTFLOW — all countries
Compiled.


chain 1:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]








chain 1:  10%|█         | 100/1000 [00:01<00:14, 61.68it/s, (Warmup)]










chain 1:  20%|██        | 200/1000 [00:02<00:08, 92.47it/s, (Warmup)]











chain 1:  30%|███       | 300/1000 [00:03<00:06, 107.61it/s, (Warmup)]

chain 1:  40%|████      | 400/1000 [00:03<00:03, 152.76it/s, (Warmup)]

chain 2: 100%|██████████| 1000/1000 [00:04<00:00, 223.79it/s, (Sampling completed)]

chain 3: 100%|██████████| 1000/1000 [00:04<00:00, 223.80it/s, (Sampling completed)]


chain 4: 100%|██████████| 1000/1000 [00:04<00:00, 223.80it/s, (Sampling completed)]


05:54:17 - cmdstanpy - INFO - CmdStan done processing.
05:54:17 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'paper_outflow.stan', line 53, column 2 to column 25)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'paper_outflow.stan', line 53, column 2 to column 25)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'paper_outflow.stan', line 53, column 2 to column 25)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'paper_outflow.stan', line 53, column 2 to column 25)
Consider re-running with show_console=True if the above output is unclear!


...

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

Rank-normalized split R-hat values satisfactory for all parameters.

Processing complete, no problems detected.


FITTING: INFLOW — all corridors
Compiled.


chain 1:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]





chain 1:  10%|█         | 100/1000 [50:19<7:34:13, 30.28s/it, (Warmup)]


chain 1:  20%|██        | 200/1000 [2:11:13<9:07:11, 41.04s/it, (Warmup)]



chain 1:  30%|███       | 300/1000 [3:34:38<8:46:49, 45.16s/it, (Warmup)]




chain 1:  40%|████      | 400/1000 [4:57:53<7:50:28, 47.05s/it, (Warmup)]


chain 1:  50%|█████     | 501/1000 [6:16:09<6:29:21, 46.82s/it, (Sampling)]






chain 1:  60%|██████    | 600/1000 [7:35:14<5:15:14, 47.29s/it, (Sampling)]


chain 1:  70%|███████   | 700/1000 [8:58:38<4:01:37, 48.32s/it, (Sampling)]


chain 1:  80%|████████  | 800/1000 [10:23:57<2:44:25, 49.33s/it, (Sampling)]


chain 1:  90%|█████████ | 900/1000 [11:47:11<1:22:33, 49.53s/it, (Sampling)]


chain 1: 100%|██████████| 1000/1000 [13:10:54<00:00, 49.76s/it, (Sampling)] 

chain 2: 100%|██████████| 1000/1000 [13:22:53<00:00, 48.17s/it, (Sampling completed)]

chain 3: 100%|██████████| 1000/1000 [13:22:53<00:00, 48.17s/it, (Samp


19:17:11 - cmdstanpy - INFO - CmdStan done processing.


19:17:24 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 73 divergent transitions (14.6%)
	Chain 1 had 426 iterations at max treedepth (85.2%)
	Chain 2 had 51 divergent transitions (10.2%)
	Chain 2 had 448 iterations at max treedepth (89.6%)
	Chain 3 had 46 divergent transitions (9.2%)
	Chain 3 had 453 iterations at max treedepth (90.6%)
	Chain 4 had 62 divergent transitions (12.4%)
	Chain 4 had 437 iterations at max treedepth (87.4%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


Done in 48186s (803.1 min)
Checking sampler transitions treedepth.
1768 of 2000 (88.40%) transitions hit the maximum treedepth limit of 12, or 2^12 leapfrog steps.
Trajectories that are prematurely terminated due to this limit will result in slow exploration.
For optimal performance, increase this limit.

Checking sampler transitions for divergences.
232 of 2000 (11.60%) transitions ended with a divergence.
These divergent transitions indicate that HMC is not fully able to explore the posterior distribution.
Try increasing adapt delta closer to 1.
If this doesn't remove all divergences, try to reparameterize the model.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01:
  kappa[1], kappa[2], kappa[3], kappa[4], kappa[5], kappa[6], kappa[7], kappa[8], kappa[9], kappa[10], kappa[11], kappa[12], kappa[13], 